# Kaggriculture: selling 300 melons returns 36% of face value

*What the engine source and 44,000 public episodes say about price, demand, and the
shed.*

This notebook reads two things and nothing else:

1. **the referee source** that ships with `kaggle-environments`, quoted with line
   numbers, and
2. **the public episode record**, via the community dataset
   [`georgymamarin/kaggriculture-episodes`](https://www.kaggle.com/datasets/georgymamarin/kaggriculture-episodes),
   which parses Kaggle's official daily episode dumps into tidy CSVs and is
   refreshed daily.

Every table and figure is produced by the cell directly above it. Nothing is copied
in from a private run. There is no agent-tuning advice here — this is about what the
rules and the record say.

**Contents**

1. What selling does to your own price
2. How much demand the town absorbs per day
3. The action and order schema: three channels, quantities, and the cap of ten
4. The shed: one hundred units, and what happens to the rest
5. Money by rating band, and where it stops mattering

> **Version note, and it matters.** The competition ladder runs
> `kaggle-environments` **1.32.4**, but Kaggle's default notebook image ships an
> older build whose market parameters are different — on 1.29.3 the same 300-melon
> sale returns 70% rather than 36%. The cell below pins the engine so every number
> here describes the game as it is actually played. If the pin fails, the notebook
> still runs; just read the version it prints and take the cells over the prose.

In [ ]:
%pip install -q "kaggle-environments==1.32.4"

In [ ]:
import math, os, sys
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (8, 4.2), "axes.grid": True, "grid.alpha": .25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})

import kaggle_environments
from kaggle_environments import make
from kaggle_environments.envs.kaggriculture import kaggriculture as K

SRC = K.__file__
SRC_LINES = open(SRC, encoding="utf-8").read().split("\n")
try:
    from importlib.metadata import version as _pkg_version
    _ver = _pkg_version("kaggle-environments")
except Exception:
    _ver = "unknown"
print("kaggle-environments", _ver)
print("referee source     ", SRC)

import inspect

def show_func(obj, title=None):
    "Print a referee function's real source, with its real line numbers."
    lines, start = inspect.getsourcelines(obj)
    name = title or obj.__name__
    print(f"--- {name}  [lines {start}-{start + len(lines) - 1}] ---")
    for i, line in enumerate(lines):
        print(f"{start + i:4d} | {line.rstrip()}")

def show_block(anchor, after=0, before=0, title=None):
    "Print the lines around the first line containing `anchor`."
    hits = [i for i, l in enumerate(SRC_LINES) if anchor in l]
    if not hits:
        print(f"!! anchor not found in this version: {anchor!r}")
        return
    i = hits[0]
    lo, hi = max(0, i - before), min(len(SRC_LINES), i + after + 1)
    print(f"--- {title or anchor}  [lines {lo + 1}-{hi}] ---")
    for j in range(lo, hi):
        print(f"{j + 1:4d} | {SRC_LINES[j].rstrip()}")

# Line numbers differ between kaggle-environments releases, so nothing below is
# hard-coded to a line range -- we look code up by symbol or by anchor text.

> Everything below is read from the installed referee printed above. If that is not
> 1.32.4, the numbers will differ from the prose — trust the cells.

## 1. What selling does to your own price

Start here, because it is the fact that reframes everything else: **there is no clock
in the price function.** Price is a pure function of how much of that product is
currently sitting in the market's inventory.

In [ ]:
show_func(K.market_price)
show_func(K._refresh_prices)
show_block("price(inv) = base", after=11, before=1,
           title="the pricing model, in the source's own words")

Each product has a base price and a neutral inventory `I0`. Below `I0` the price
rises along one curve; above it, it falls along another. The curves are chosen per
product, and the amplitude is calibrated so that moving `T` units shifts the price by
`target × base`.

Those shapes are the whole story. Melon's glut curve is `sq` with a target of 3.6 —
quadratic and brutal. Strawberry's is `linear` at 1.6. Wheat's is `log` at 0.2, which
barely moves at all.

In [ ]:
params = pd.DataFrame(K.MARKET_PARAMS).T
params[["base", "T", "below_func", "below_target", "above_func", "above_target"]]

In [ ]:
I0 = K.MARKET_I0
offsets = np.arange(0, 601)

fig, ax = plt.subplots()
for item, colour in [("MELON", "#c05621"), ("STRAWBERRY", "#b83280"),
                     ("MILK", "#2b6cb0"), ("WHEAT", "#2f855a")]:
    ax.plot(offsets, [K.market_price(item, I0 + int(o)) for o in offsets],
            label=item, color=colour, lw=2)
ax.set_xlabel("units sold into the market (inventory above neutral)")
ax.set_ylabel("unit price")
ax.set_title("What selling does to your own price")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

Now the part that catches people out. A sell order is filled **one unit at a time**,
and the price is re-quoted for every unit (we will see that loop in section 3). So a
big sale walks its own price down as it fills — you do not get the price you saw when
you placed it.

In [ ]:
def fill_value(item, qty, inventory=None):
    "Total proceeds from selling `qty` units one at a time, and the naive figure."
    inv = K.MARKET_I0 if inventory is None else inventory
    fills = [K.market_price(item, inv + i) for i in range(qty)]
    return sum(fills), qty * fills[0], fills[0], fills[-1]

rows = []
for item in ["MELON", "STRAWBERRY", "MILK", "TOMATO", "CARROT", "WHEAT"]:
    for qty in (50, 100, 300):
        got, naive, first, last = fill_value(item, qty)
        rows.append({"product": item, "units": qty, "first unit": first,
                     "last unit": last, "actually got": got,
                     "naive (qty x first)": naive,
                     "% of naive": round(100 * got / naive)})
fills = pd.DataFrame(rows)
fills.pivot(index="product", columns="units", values="% of naive").loc[
    ["MELON", "STRAWBERRY", "MILK", "TOMATO", "CARROT", "WHEAT"]]

In [ ]:
fills[fills["units"] == 300].set_index("product")[
    ["first unit", "last unit", "actually got", "naive (qty x first)", "% of naive"]]

**Selling 300 melons returns about 36% of what the sticker price suggests** — the
first unit goes for 250 and the last for 1. The same 300 units of wheat lose almost
nothing. Melon is simultaneously the most valuable product in the game and the one
that punishes volume hardest, and those two facts are the same fact.

The practical reading: for melon and strawberry, *how much you sell at once* matters
more than the headline price. For wheat it barely matters at all.

In [ ]:
fig, ax = plt.subplots()
for item, colour in [("MELON", "#c05621"), ("STRAWBERRY", "#b83280"),
                     ("MILK", "#2b6cb0"), ("WHEAT", "#2f855a")]:
    qs = np.arange(10, 401, 10)
    keep = [100 * fill_value(item, int(q))[0] / fill_value(item, int(q))[1] for q in qs]
    ax.plot(qs, keep, label=item, color=colour, lw=2)
ax.axhline(100, ls="--", color="grey", lw=1)
ax.set_xlabel("units sold in one order")
ax.set_ylabel("% of the naive value you actually receive")
ax.set_title("Big orders pay less per unit, and by wildly different amounts")
ax.set_ylim(0, 105); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 2. How much demand the town absorbs per day

If selling pushes the price down, something must push it back up, or the market would
only ever decay. That something is the town: it removes goods from the market
inventory on a fixed schedule.

In [ ]:
show_func(K._town_consume)
show_block("SHOPS = {", after=14,
           title="SHOPS, TOWN_CENTER_PRODUCTS, TOWN_CENTER_DEMAND_SCHEDULE")

Reading it off:

* Every `townShopSellInterval` steps (default **4**, so **6 times a day**) each
  *unlocked* shop removes one unit of each product it uses — **two** units if the
  shop sells only a single product.
* Every `townCenterSellInterval` steps (default **12**, so **twice a day**) the town
  centre removes `center_mult` units of **every** product except fertilizer, and
  `center_mult` steps up with the day: **1** before day 10, **2** from day 10, **4**
  from day 20.
* Shops unlock one at a time, every `townShopUnlockInterval` days (default **3**),
  chosen at random from those not yet open.

So absorbed demand per product per day is
`6 × (shop multipliers for that product) + 2 × center_mult`.

In [ ]:
CFG = make("kaggriculture", debug=False).configuration
SHOP_SELL    = int(CFG.get("townShopSellInterval", 4))
CENTER_SELL  = int(CFG.get("townCenterSellInterval", 12))
UNLOCK_EVERY = int(CFG.get("townShopUnlockInterval", 3))
TPD          = int(CFG.get("turnsPerDay", 24))
print(f"shop sell every {SHOP_SELL} steps | town centre every {CENTER_SELL} "
      f"| a shop unlocks every {UNLOCK_EVERY} days | {TPD} turns/day")

def center_mult(day):
    return next(m for thr, m in K.TOWN_CENTER_DEMAND_SCHEDULE if day >= thr)

def absorbed_per_day(day, unlocked):
    "Units of each product the town removes on `day`, given the open shops."
    per = Counter()
    for shop in unlocked:
        products = K.SHOPS[shop]
        mult = 2 if len(products) == 1 else 1
        for p in products:
            per[p] += (TPD // SHOP_SELL) * mult
    for p in K.TOWN_CENTER_PRODUCTS:
        per[p] += (TPD // CENTER_SELL) * center_mult(day)
    return per

tiers = {label: absorbed_per_day(day, list(K.SHOPS)) for day, label in
         [(0, "day 0-9"), (10, "day 10-19"), (20, "day 20+")]}
demand = pd.DataFrame(tiers).fillna(0).astype(int)
demand.sort_values(demand.columns[-1], ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.6))
demand.sort_values(demand.columns[-1]).plot.barh(
    ax=ax, width=.8, color=["#bee3f8", "#63b3ed", "#2b6cb0"])
ax.set_xlabel("units absorbed per day (all shops open)")
ax.set_title("Town demand per product, by season tier")
ax.legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.show()

* **Wheat is the most-wanted crop** simply because it appears in the most shop
  recipes — nothing intrinsic to wheat.
* **Melon is the least absorbed**, by a wide margin: it is in no shop recipe at all,
  so its only buyer is the town centre. Combine that with section 1's quadratic glut
  curve and melon becomes the clearest example in the game of a product whose price
  you can destroy and then have to wait out.
* **Demand roughly doubles over a season** for every product as the centre multiplier
  goes 1 → 2 → 4 and shops keep opening. The same price late in the game is absorbing
  far more supply than it was early.

The table assumes all eight shops are open. Early on only a few are, and which ones is
random — so two seeds can hand you quite different markets.

In [ ]:
# What one day of town demand is worth at neutral inventory, day 20+.
per_day = absorbed_per_day(20, list(K.SHOPS))
pd.DataFrame([{"product": p, "absorbed/day": per_day[p],
               "price at I0": K.market_price(p, I0),
               "value of that drain": per_day[p] * K.market_price(p, I0)}
              for p in ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY",
                        "MELON", "EGG", "MILK", "WOOL"]]
            ).set_index("product").sort_values("value of that drain",
                                               ascending=False)

## 3. The action and order schema

An action is a dict with **three independent channels**, and the third one is free.

In [ ]:
show_block('action.get("farmer"', before=2, after=26,
           title="interpreter: how an action is unpacked")

* `farmer` — one action for your farmer.
* `hands` — a list, one action per hired hand.
* `market` — a list of orders.

Farmer and hand actions each cost that unit its turn. Market orders cost **no unit
turns at all** — they are processed separately in `_process_market`. Buying a product
is free in turns; moving it from the shed to where it is needed is not.

In [ ]:
show_block("q[:max_orders]", before=2, after=7, title="the ten-order cap")
show_func(K._parse_order, "_parse_order: orders carry a quantity")

Two details that matter more than they look:

* **The cap is ten orders per turn, and it is a slice.** `q[:max_orders]` — the
  eleventh order is dropped silently, with no error. If you build orders in a loop,
  the tail of that loop may simply never happen.
* **An order carries a quantity.** `["SELL", "WHEAT", 60]` is *one* order of the ten,
  not sixty. So the cap is far less restrictive than it looks, as long as you
  aggregate.

And here is the loop that fills them, which is where section 1's per-unit repricing
actually lives.

In [ ]:
show_block("quoted = [None, None]", before=6, after=37,
           title="_process_market: per-unit lockstep")

Note `ostate["remaining"] -= 1`: one unit per pass, re-quoted each time against the
current inventory. That is the mechanism behind the melon table — the order is not
filled at one price, it is filled at three hundred of them.

Worth noticing while we are here: within each unit, both players are quoted *before*
either trade commits (line 589). The market does not reward going first.

## 4. The shed: one hundred units, and what happens to the rest

Everything you harvest lands in the harvesting unit's own inventory. To sell it, it
has to reach the shed — and the shed is small.

In [ ]:
print("shedCapacity:", CFG.get("shedCapacity", 100))
show_func(K._shed_access_tiles, "_shed_access_tiles: the shed has four doors")
show_func(K._drop_inventories_to_shed,
          "_drop_inventories_to_shed: end of day")

Three constraints stack here:

* The shed holds **100 units in total** — one shared pool across every product, not
  100 of each.
* `PICKUP` and `DROP` only work from **four tiles**, the inner corners around the
  shed. Everything you bank has to walk to one of those doors.
* At the end of every day, whatever is still in a unit's inventory is pushed into the
  shed — and the docstring is blunt about the rest: **overflow is discarded.** No
  error, no warning, no partial credit.

That last one is easy to verify directly.

In [ ]:
# 80 units already banked, two hands carrying 30 each: what survives the night?
private = {"shed": {"WHEAT": 80},
           "inventories": [{"MELON": 30}, {"MELON": 30}],
           "seeds": {}}
before = (sum(private["shed"].values())
          + sum(n for inv in private["inventories"] for n in inv.values()))
K._drop_inventories_to_shed(private, int(CFG.get("shedCapacity", 100)))
after = sum(private["shed"].values())
print("held before the drop:", before)
print("in the shed after   :", after, private["shed"])
print("silently destroyed  :", before - after)

Twenty melons survive; forty vanish. Nothing in the observation tells you it happened.

The throughput consequence is the useful one. The shed is not just storage, it is a
**rate limit**: between sales you cannot bank more than 100 units, no matter how many
tiles you are working or how many hands you have. Past that point, extra harvest is
not worth less — it is worth nothing.

In [ ]:
# How much a full shed is worth, and how quickly a day's harvest can exceed it.
rows = []
for item in ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL"]:
    got, naive, first, last = fill_value(item, 100)
    rows.append({"product": item, "100 units, sold at once": got,
                 "per unit": round(got / 100, 1),
                 "% of naive": round(100 * got / naive)})
pd.DataFrame(rows).set_index("product").sort_values(
    "100 units, sold at once", ascending=False)

A shed full of melon is worth several times a shed full of wheat even after the price
walk — but you can only hold one shedful at a time, and the town absorbs melon eight
units a day at best. Sections 1, 2 and 4 are really one constraint viewed three ways.

## 5. Money by rating band

Now the record rather than the rules. The official dataset carries one row per
(episode, seat) with that seat's final money and the rating it held, so we can ask
what actually separates a 1,000-rated agent from a 3,000-rated one.

One filter matters: the dataset mixes ladder games with self-play validation runs, and
the README says to drop the latter for strength analysis.

In [ ]:
import glob

def find_dataset():
    "Locate whichever attached dataset carries episodes.csv (any depth)."
    for pattern in ("/kaggle/input/**/episodes.csv", "../input/**/episodes.csv",
                    "tmp/dsnew/episodes.csv", "tmp/ds/episodes.csv"):
        hits = sorted(glob.glob(pattern, recursive=True))
        if hits:
            return os.path.dirname(hits[0])
    return None

BASE = find_dataset()
if BASE is None:
    print("episodes.csv not found. Visible inputs:")
    for p in sorted(glob.glob("/kaggle/input/*")):
        print("   ", p, sorted(os.listdir(p))[:6])
    raise SystemExit(
        "Attach georgymamarin/kaggriculture-episodes and re-run this section.")
print("dataset:", BASE)

episodes = pd.read_csv(os.path.join(BASE, "episodes.csv"))
feats    = pd.read_csv(os.path.join(BASE, "episode_features.csv"))
print(f"{len(episodes):,} episodes, {len(feats):,} agent-episode rows")
print(episodes["type"].value_counts().to_string())

In [ ]:
public = set(episodes.loc[episodes["type"] == "EPISODE_TYPE_PUBLIC", "episode_id"])
feats  = feats[feats["episode_id"].isin(public)].copy()

ratings = pd.concat([
    episodes[["episode_id", "rating_0"]].rename(columns={"rating_0": "rating"}).assign(seat=0),
    episodes[["episode_id", "rating_1"]].rename(columns={"rating_1": "rating"}).assign(seat=1),
])
df = feats.merge(ratings, on=["episode_id", "seat"], how="inner").dropna(
    subset=["rating", "final_money"])
print(f"{len(df):,} ladder agent-episodes with both a rating and a final bank")

EDGES  = [0, 750, 1250, 1750, 2250, 2750, 10 ** 9]
LABELS = ["~500", "~1000", "~1500", "~2000", "~2500", "3000+"]
df["band"] = pd.cut(df["rating"], bins=EDGES, labels=LABELS, right=False)

bands = df.groupby("band", observed=True)["final_money"].agg(
    n="size", q1=lambda s: s.quantile(.25), median="median",
    q3=lambda s: s.quantile(.75)).round(0).astype(int)
bands

In [ ]:
fig, ax = plt.subplots()
order = [b for b in LABELS if b in bands.index]
# NB: matplotlib renamed boxplot's `labels` to `tick_labels` in 3.9, so we set the
# tick labels ourselves and stay version-agnostic.
ax.boxplot([df.loc[df["band"] == b, "final_money"] for b in order],
           showfliers=False, widths=.6,
           medianprops=dict(color="#c05621", lw=2))
ax.set_xticks(range(1, len(order) + 1)); ax.set_xticklabels(order)
ax.set_xlabel("rating band"); ax.set_ylabel("final money (coins)")
ax.set_title("Money stops separating agents early, and then stays flat")
ax.yaxis.set_major_formatter(lambda v, p: f"{v:,.0f}")
plt.tight_layout(); plt.show()

In [ ]:
pd.DataFrame({"median money": bands["median"],
              "gain over previous band": bands["median"].diff()}
             ).fillna(0).astype(int)

**The interesting part is where the curve stops.** Almost the entire climb happens
below the ~1000 band. After that it flattens hard: on the snapshot this notebook ran
against, the median goes 34k at ~500, then 83k at ~1000 — and then 89k, 86k, 92k, 95k
across ~1500, ~2000, ~2500 and 3000+.

Put plainly: **the first jump roughly triples your money, and the next two thousand
rating points add about 14%.** Re-run the cell above on today's data before quoting
those figures — the bands shift as the ladder matures — but the shape has been the
same every time we have looked.

That is worth knowing before spending a week optimising output. Getting to roughly the
~1000 band *is* an earning problem. Past it, the record says the remaining ratings are
decided by something this column cannot see — and the constraints in sections 1, 2 and
4 suggest why there is a ceiling on banking at all: the shed caps the stock you can
hold, the town caps the flow it will absorb, and the price curve punishes you for
pushing past either.

---

### Companion notebook

This one is about the mechanism from the inside. A companion,
[Does a farming game get supply and demand right?](https://www.kaggle.com/code/xreina8/does-a-farming-game-get-supply-and-demand-right),
takes the same pricing rule and holds it against a real commodity market — fifteen
years of US honey production and prices — to ask whether the price collapse above is
realistic. Short answer: at small volumes yes, past a quarter of a field-season it
exaggerates by three to eighteen times.

A third, [What decides rating once money stops mattering?](https://www.kaggle.com/code/xreina8/what-decides-rating-once-money-stops-mattering), picks up where
section 5 below stops: money saturates around a rating of 1000, so what decides the
two thousand points above it?

### Reproducing this

Attach `kaggle/kaggriculture-episodes` (any daily snapshot) and run top to bottom. The
engine sections need no data — they read the referee that ships with
`kaggle-environments`, so they stay correct for whatever version you have installed,
and will visibly change if the rules do.